# Advanced RAG System

## 1. Import Dependencies

In [74]:
import json
import sys
import os
from pathlib import Path
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from sentence_transformers import CrossEncoder
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.memory import ConversationBufferMemory

## 2. Setup and Configuration

In [75]:
# Paths
RAW_DATA_PATH = "data/raw"
PROCESSED_DIR = Path("data/processed")
ARTIFACTS_DIR = Path("artifacts")


# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings,
    validate_api_keys,
    print_config_summary
)

from src.services.cost_tracking import (
    count_tokens,
    analyze_query_cost
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


📂 Working directory: /Users/mircofernando/Documents/AI Engineering/RAG/Mini-Project/uber_rag
✅ Config loaded:
  LLM: openrouter (openai/gpt-4o-mini)
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


/Users/mircofernando/Documents/AI Engineering/RAG/Mini-Project/uber_rag/src/services/llm_services.py:375: UserWarning: ⚠️  GROQ_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
/Users/mircofernando/Documents/AI Engineering/RAG/Mini-Project/uber_rag/src/services/llm_services.py:375: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


## 3. Model Initialization

In [79]:
llm = get_llm(config)
embeddings = get_text_embeddings(config)

# Reranker (specific to this notebook - now from config)
rerank_cfg = config.get('rerank_settings', {})
reranker_model = rerank_cfg.get('model', "cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoder(reranker_model)

print(f"✅ LLM: {config['llm_provider']} / {config.get('openrouter_model', config.get('llm_model'))}")
print(f"✅ Embeddings: {config['text_emb_model']}")
print(f"✅ Reranker: {reranker_model}")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ LLM API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")

✅ LLM: openrouter / gpt-4o-mini
✅ Embeddings: sentence-transformers/all-MiniLM-L6-v2
✅ Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

🔍 Testing LLM API connection...
✅ LLM API verified: API working!


## 4. Database Initialization 

In [80]:
import weaviate
import weaviate.classes.config as wvc
from weaviate.classes.init import Auth


# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WCD_URL"),
    auth_credentials=Auth.api_key(os.getenv("WCD_API_KEY")),
    headers={"X-HuggingFace-Api-Key": os.getenv("HF_API_KEY")}
)

# Define the Schema
if not client.collections.exists("UberFinancials"):
    collection = client.collections.create(
        name="UberFinancials",
        # Dense Vectorizer
        vectorizer_config=wvc.Configure.Vectorizer.text2vec_huggingface(
            model="sentence-transformers/all-MiniLM-L6-v2"
        ),
        # Cross-Encoder Reranker
        reranker_config=wvc.Configure.Reranker.transformers(),
        # Properties
        properties=[
            wvc.Property(name="content", data_type=wvc.DataType.TEXT),
            wvc.Property(name="source", data_type=wvc.DataType.TEXT)
        ]
    )
    print("✅ Schema created successfully!")



## 5. Data Ingestion

In [ ]:
def ingest_uber_data(json_file_path: str):
    print("Starting ingestion process...")
    
    with open(json_file_path, "r", encoding="utf-8") as file:
        chunks = json.load(file)
    
    with client.batch.dynamic() as batch:
            for i, chunk in enumerate(chunks):
                
                text_content = chunk.get("page_content", "")
                metadata = chunk.get("metadata", {})
                # Format citation metadata
                page_num = metadata.get("page", "Unknown")
                source_name = metadata.get("source", "uber.pdf").split("/")[-1]
                citation = f"{source_name} - Page {page_num}"
                
                batch.add_object(
                    collection="UberFinancials",
                    properties={
                        "content": text_content,
                        "source": citation
                    }
                )
                
                if i % 100 == 0 and i > 0:
                    print(f"Queued {i} chunks...")

    if len(client.batch.failed_objects) > 0:
        print(f"⚠️ Warning: {len(client.batch.failed_objects)} objects failed.")
    else:
        print("✅ Ingestion complete! The database is primed.")

ingest_uber_data(ARTIFACTS_DIR / "data_factory" / "chunks.json")


{'message': 'Failed to send all objects in a batch of 48', 'error': "WeaviateClosedClientError('The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!')"}
{'message': 'Failed to send 48 objects in a batch of 48. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send all objects in a batch of 48', 'error': "WeaviateClosedClientError('The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!')"}
{'message': 'Failed to send 48 objects in a batch of 48. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send all objects in a batch of 48', 'error': "WeaviateClosedClientError('The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!')"}
{'message': 'Failed to send 48 objects in a batch of 48. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'me

Starting ingestion process...
Queued 100 chunks...
Queued 200 chunks...
Queued 300 chunks...
Queued 400 chunks...
Queued 500 chunks...


{'message': 'Failed to send all objects in a batch of 14', 'error': "WeaviateClosedClientError('The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!')"}
{'message': 'Failed to send 14 objects in a batch of 14. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}


⚠️ Warning: 542 objects failed.


## 7. Hybrid Retrieval (BM25 + Dense)

In [81]:
import weaviate.classes.query as wq

def ensure_weaviate_client():
    try:
        if not client.is_connected():
            client.connect()
    except Exception:
        client.connect()

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="input",
    output_key="output",
    return_messages=False,
 )

def hybrid_retrieval(question: str, limit: int = 20) -> List[Dict]:
    """Return structured candidate docs for reranking."""
    ensure_weaviate_client()
    collection = client.collections.get("UberFinancials")

    response = collection.query.hybrid(
        query=question,
        alpha=0.5,
        limit=limit,
        fusion_type=wq.HybridFusion.RANKED,
    )

    candidates = []
    for obj in response.objects:
        candidates.append({
            "content": obj.properties.get("content", ""),
            "source": obj.properties.get("source", "Unknown source"),
        })

    return candidates

## 8. Cross-Encoder Reranking 

In [82]:
def rerank(query: str, results: List[Dict], top_k: int = 5) -> List[Dict]:
    """Rerank a small candidate set with the cross-encoder and keep only the best matches."""
    if not results:
        return []

    pairs = [[query, item["content"]] for item in results]
    scores = reranker.predict(pairs)

    ranked = []
    for item, score in zip(results, scores):
        ranked.append({
            **item,
            "rerank_score": float(score),
        })

    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_k]

## Complete Hybrid RAG Pipeline

In [92]:
template = """
You are a highly accurate financial analyst. Answer the user's question using ONLY the provided context from the Uber 2024 Annual Report.
If the answer is not contained in the context, explicitly state that you do not know.

Previous Conversation Summary:
{chat_history}

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

def format_context(docs: List[Dict]) -> str:
    return "\n\n---\n\n".join(
        f"[Source: {doc['source']}]\n{doc['content']}" for doc in docs
    )

def retrieve_and_rerank(question: str) -> str:
    candidates = hybrid_retrieval(question)
    top_docs = rerank(question, candidates, top_k=5)
    return format_context(top_docs), top_docs  # Return both formatted context and top docs for later use

rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: retrieve_and_rerank(x["question"])),
        chat_history=RunnableLambda(lambda x: x.get("chat_history", "")),
    )
    | prompt
    | llm
    | StrOutputParser()
 )

def chat_with_uber(user_input: str) -> str:
    """Handles memory loading, chain execution, and memory saving."""
    history = memory.load_memory_variables({}).get("chat_history", "")
    answer = rag_chain.invoke({
        "question": user_input,
        "chat_history": history,
    })
    
    context, top_docs = retrieve_and_rerank(user_input)
    prompt_text = template.format(chat_history=history, context=context, question=user_input)

    response = llm.invoke(prompt_text)

    answer = response.content if hasattr(response, "content") else str(response)
    
    try:
        memory.save_context({"input": user_input}, {"output": answer})
    except Exception:
        pass

    return answer, top_docs

In [94]:
query = "What was Uber's total revenue in 2024?"
answer, docs = chat_with_uber(query)
print(answer)
print("\n---\n")
    
# Question 2 (The LLM will now remember the context of Question 1!)
print(chat_with_uber("Did that represent an increase from the previous year?"))
    

Uber's total revenue in 2024 was $43,978 million.

---

("Yes, Uber's total revenue in 2024 represented an increase from the previous year, as it was $43,978 million in 2024 compared to $37,281 million in 2023, reflecting an increase of $6.7 billion or 18% year-over-year.", [{'content': 'Revenue \n \n \nYear Ended December 31,  \n% Change \n(In millions, except percentages) \n \n2023 \n \n2024 \n \nRevenue \n $ \n37,281  $ \n43,978  \n 18 % \n2024 Compared to 2023 \nRevenue increased $6.7 billion, or 18% year-over-year, primarily attributable to an increase in Gross Bookings of 18%. The \nincrease in Gross Bookings was primarily driven by an increase in Mobility and Delivery Trip volumes. The increase in revenue was \npartially offset by business model changes in some countries that classified certain sales and marketing costs as contra revenue, which \nnegatively impacted revenue by $863 million and $713 million across Mobility and Delivery, respectively. \nCost of Revenue, Exclusive 

In [100]:
print(answer)

print(docs[0]['content'])


Uber's total revenue in 2024 was $43,978 million.
75 
UBER TECHNOLOGIES, INC. 
CONSOLIDATED STATEMENTS OF OPERATIONS 
(In millions, except share amounts which are reflected in thousands, and per share amounts) 
 
 
Year Ended December 31, 
 
 
2022 
 
2023 
 
2024 
Revenue 
 $ 31,877  $ 37,281  $ 43,978  
Costs and expenses 
  
  
  
Cost of revenue, exclusive of depreciation and amortization shown separately below 
  
19,659   
22,457   
26,651  
Operations and support 
  
2,413   
2,689   
2,732  
Sales and marketing 
  
4,756   
4,356   
4,337  
Research and development 
  
2,798   
3,164   
3,109  
General and administrative 
  
3,136   
2,682   
3,639  
Depreciation and amortization 
  
947   
823   
711  
Total costs and expenses 
  
33,709   
36,171   
41,179  
Income (loss) from operations 
  
(1,832)  
1,110   
2,799  
Interest expense 
  
(565)  
(633)  
(523) 
Other income (expense), net 
  
(7,029)  
1,844   
1,849  
Income (loss) before income taxes and income (loss) from 

## RAG Evaluation

In [101]:
cost = analyze_query_cost(docs[0]['content'] + query, answer)  # For cost tracking

TypeError: unsupported operand type(s) for /: 'str' and 'int'